# Credit Card Fraud Detection

End-to-end fraud detection pipeline combining **transactional features** and **graph-based network features**, built on a BankSim-style dataset.

**Pipeline:**
1. Load & clean data
2. Exploratory data analysis
3. Feature engineering (tabular + graph)
4. Train/test split & class-imbalance handling (SMOTE)
5. Model training (XGBoost)
6. Evaluation (Precision, Recall, PR-AUC, ROC-AUC)
7. Threshold tuning
8. Cross-validation
9. Explainability (SHAP)
10. Export results for Power BI dashboard


## 1. Setup

Install and import required libraries.

In [ ]:
# Uncomment to install dependencies
# pip install pandas numpy scikit-learn imbalanced-learn xgboost networkx shap joblib --break-system-packages

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_curve,
    average_precision_score, roc_auc_score, precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import joblib

pd.set_option('display.max_columns', None)


## 2. Load Data

- `df1`: transaction-level data (customer, merchant, category, amount, fraud label)
- `df2`: the same transactions represented as a customer→merchant graph edge list


In [ ]:
df1 = pd.read_csv('data/raw_data_1.csv')
df2 = pd.read_csv('data/raw_data_2.csv')

print(df1.shape, df2.shape)
df1.head()


In [ ]:
df2.head()

## 3. Data Cleaning

Raw values are stored with stray quotes (e.g. `'C1093826151'`), and `age` uses `'U'` for unknown. Clean both before any feature engineering.


In [ ]:
# Strip stray quote characters from all string/object columns
for frame in (df1, df2):
    str_cols = frame.select_dtypes(include='object').columns
    for col in str_cols:
        frame[col] = frame[col].str.strip("'")

# 'U' = unknown age -> encode as -1 (tree models can split this into its own branch)
df1['age'] = df1['age'].replace('U', '-1')

# Enforce correct dtypes
df1 = df1.astype({
    'age': int, 'zipcodeOri': int, 'zipMerchant': int,
    'amount': float, 'fraud': int
})
df2 = df2.astype({'Weight': float, 'fraud': int})

print(df1.dtypes)
print(df1.isnull().sum())
print(f"Fraud rate: {df1['fraud'].mean() * 100:.2f}%")


## 4. Exploratory Data Analysis

Check how fraud relates to transaction amount and merchant category before engineering features.


In [ ]:
sns.boxplot(x='fraud', y='amount', data=df1)
plt.title('Transaction Amount: Fraud vs Not Fraud')
plt.show()


In [ ]:
df1.groupby('category')['fraud'].mean().sort_values(ascending=False)


In [ ]:
df1['gender'].value_counts()


## 5. Feature Engineering — Tabular

Fraud is best captured by **behavioral deviation**, not raw IDs. Engineer per-customer and per-merchant activity features.


In [ ]:
df1['customer_txn_count'] = df1.groupby('customer')['customer'].transform('count')
df1['merchant_txn_count'] = df1.groupby('merchant')['merchant'].transform('count')
df1['customer_avg_amount'] = df1.groupby('customer')['amount'].transform('mean')
df1['amount_deviation'] = df1['amount'] - df1['customer_avg_amount']

# One-hot encode categoricals
df1_encoded = pd.get_dummies(df1[['gender', 'category']], drop_first=True)
df1_encoded.head()


## 6. Feature Engineering — Graph (df2)

Build a customer↔merchant graph from `df2` and extract structural features (degree, PageRank). Fraud often shows up as unusual connectivity patterns that a flat table can't capture.


In [ ]:
G = nx.from_pandas_edgelist(df2, 'Source', 'Target', edge_attr='Weight', create_using=nx.Graph())

degree = dict(G.degree())
pagerank = nx.pagerank(G, weight='Weight')

df1['customer_degree'] = df1['customer'].map(degree)
df1['merchant_degree'] = df1['merchant'].map(degree)
df1['customer_pagerank'] = df1['customer'].map(pagerank)

df1[['customer', 'merchant', 'customer_degree', 'merchant_degree', 'customer_pagerank']].head()


## 7. Assemble Final Feature Matrix

In [ ]:
features = [
    'age', 'amount', 'customer_txn_count', 'merchant_txn_count',
    'amount_deviation', 'customer_degree', 'merchant_degree', 'customer_pagerank'
]

X = pd.concat([df1[features], df1_encoded], axis=1).fillna(0)
y = df1['fraud']

X.shape, y.shape


## 8. Train/Test Split

Stratified split preserves the same fraud ratio in both sets — critical with ~1-2% fraud.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape


## 9. Handle Class Imbalance (SMOTE)

Applied to the **training set only**, to avoid leaking synthetic points derived from test data.


In [ ]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(y_train.value_counts())
print(y_train_res.value_counts())


## 10. Train Model (XGBoost)

In [ ]:
model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    eval_metric='aucpr', random_state=42
)
model.fit(X_train_res, y_train_res)


## 11. Evaluate Model

Accuracy is misleading on imbalanced data — use precision, recall, F1, PR-AUC, and ROC-AUC instead.


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("PR-AUC:", average_precision_score(y_test, y_proba))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))


## 12. Threshold Tuning

Pick an operating threshold based on the business need (e.g. recall >= 0.9), rather than the default 0.5 cutoff.


In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

idx = np.argmax(recalls >= 0.9)
best_threshold = thresholds[idx]
print(f"Threshold for recall >= 0.90: {best_threshold:.4f}")


## 13. Cross-Validation

Confirm results are stable across folds, not a fluke of one split.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='average_precision')
print(f"CV PR-AUC: {scores.mean():.4f} +/- {scores.std():.4f}")


## 14. Explainability (SHAP)

Understand which features drive each prediction — required for auditability in real fraud systems.


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)


## 15. Save Model

In [ ]:
joblib.dump(model, 'fraud_model.pkl')
joblib.dump(list(X.columns), 'model_features.pkl')


## 16. Export Results for Power BI Dashboard

Export transaction-level predictions, summary metrics, confusion matrix, and SHAP importance as CSVs for the presentation layer.


In [ ]:
# Transaction-level predictions
df1['predicted_fraud'] = model.predict(X)
df1['fraud_probability'] = model.predict_proba(X)[:, 1]
df1.to_csv('outputs/fraud_transactions_final.csv', index=False)

# Model evaluation metrics
metrics = pd.DataFrame({
    'metric': ['Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC'],
    'value': [
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_proba),
        average_precision_score(y_test, y_proba)
    ]
})
metrics.to_csv('outputs/model_metrics.csv', index=False)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Not Fraud', 'Actual: Fraud'],
    columns=['Predicted: Not Fraud', 'Predicted: Fraud']
)
cm_df.to_csv('outputs/confusion_matrix.csv')

# SHAP feature importance (magnitude + direction)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
mean_signed_shap = shap_values.mean(axis=0)

shap_importance = pd.DataFrame({
    'feature': X_test.columns,
    'mean_abs_shap': mean_abs_shap,
    'mean_signed_shap': mean_signed_shap
}).sort_values('mean_abs_shap', ascending=False)

shap_importance.to_csv('outputs/shap_feature_importance.csv', index=False)

shap_importance.head(10)
